# 6. Consultas Finales

Propósito: Consultas de negocio sobre los datos gold.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: c:\Users\user\Downloads\EP-GDM-G6
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot


In [2]:
from pyspark.sql import SparkSession, functions as F
from app.utils.spark import SparkClient

# Reutiliza la sesion activa del kernel; si no hay, crea una con la config
# Windows-correcta de SparkClient (rutas nativas Hadoop, memoria del driver).
spark = SparkSession.getActiveSession() or SparkClient().get_session()
gold_path = "data/gold"
silver_path = "data/silver"

In [3]:
# Recaudación por año
spark.sql(f"""
  SELECT dc.Anio, SUM(mi.MontoRecaudado) as TotalRecaudado
  FROM parquet.`{gold_path}/MART_INGRESOS_GEOGRAFICO.parquet` mi
  JOIN parquet.`{gold_path}/DIM_CALENDARIO.parquet` dc ON mi.AnioMes = dc.AnioMes
  GROUP BY dc.Anio ORDER BY dc.Anio
""").toPandas()

,Anio,TotalRecaudado
0,2021,41015584142.70
1,2022,45705723223.25
2,2023,42540041644.62
3,2024,44935373340.16


In [12]:
# Lookup SecEjec -> ubigeo + provincia + distrito (1:1) desde silver + DIM_GEOGRAFIA
fact = spark.read.parquet(f"{silver_path}/FACT_INGRESO.parquet")
ejec = spark.read.parquet(f"{silver_path}/DIM_EJECUTORA.parquet")
geo  = spark.read.parquet(f"{gold_path}/DIM_GEOGRAFIA.parquet")

(
    fact.select("IdEjecutora", "IdUbigeo").distinct()
        .join(ejec.select("IdEjecutora", "SEC_EJEC"), "IdEjecutora")
        .join(geo.select("IdUbigeo", "Ubigeo", "Provincia", "Distrito"), "IdUbigeo")
        .select(F.col("SEC_EJEC").cast("int").alias("SecEjec"), "Ubigeo", "Provincia", "Distrito")
        .createOrReplaceTempView("sec_ubigeo")
)

spark.sql(f"""
  SELECT su.Ubigeo, me.SecEjec, me.Ejecutora, me.Categoria,
         me.Departamento, su.Provincia, su.Distrito,
         SUM(me.MontoRecaudado) AS TotalRecaudado
  FROM parquet.`{gold_path}/MART_INGRESOS_EJECUTORA.parquet` me
  LEFT JOIN sec_ubigeo su ON me.SecEjec = su.SecEjec
  WHERE upper(me.Ejecutora) LIKE '%SANTA ROSA%'
  GROUP BY su.Ubigeo, me.SecEjec, me.Ejecutora, me.Categoria,
           me.Departamento, su.Provincia, su.Distrito
  ORDER BY TotalRecaudado DESC
""").toPandas()


,Ubigeo,SecEjec,Ejecutora,Categoria,Departamento,Provincia,Distrito,TotalRecaudado
0,050507,300477,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,E,AYACUCHO,LA MAR,SANTA ROSA,65711552.84
1,120808,301115,MUNICIPALIDAD DISTRITAL DE SANTA ROSA DE SACCO,D,JUNIN,YAULI,SANTA ROSA DE SACCO,64539802.33
2,060812,300634,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,G,CAJAMARCA,JAEN,SANTA ROSA,56505690.49
3,150139,301288,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,C,LIMA,LIMA,SANTA ROSA,51786658.71
4,210808,301670,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,F,PUNO,MELGAR,SANTA ROSA,39382035.87
5,100705,301873,MUNICIPALIDAD DISTRITAL DE SANTA ROSA DE ALTO ...,F,HUANUCO,MARANON,SANTA ROSA DE ALTO YANAJANCA,30338640.06
6,210504,301643,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,G,PUNO,EL COLLAO,SANTA ROSA,29148844.25
7,150407,301310,MUNICIPALIDAD DISTRITAL DE SANTA ROSA DE QUIVES,F,LIMA,CANTA,SANTA ROSA DE QUIVES,27606386.06
8,140114,301225,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,E,LAMBAYEQUE,CHICLAYO,SANTA ROSA,23718962.70
9,220304,301716,MUNICIPALIDAD DISTRITAL DE SANTA ROSA,G,SAN MARTIN,EL DORADO,SANTA ROSA,20158269.94


In [10]:
# Top 10 ejecutoras por recaudación
mart_ejec = spark.read.parquet(f"{gold_path}/MART_INGRESOS_EJECUTORA.parquet")
mart_ejec.groupBy("Ejecutora", "Departamento").agg(F.sum("MontoRecaudado").alias("Total")).orderBy(F.col("Total").desc()).show(10)

+--------------------+--------------------+--------------+
|           Ejecutora|        Departamento|         Total|
+--------------------+--------------------+--------------+
|MUNICIPALIDAD MET...|                LIMA|10663739861.73|
|MUNICIPALIDAD DIS...|              ANCASH| 4240983347.62|
|MUNICIPALIDAD DIS...|               CUSCO| 2216652500.87|
|MUNICIPALIDAD PRO...|              ANCASH| 1826922611.45|
|MUNICIPALIDAD PRO...|PROVINCIA CONSTIT...| 1752186749.63|
|MUNICIPALIDAD DIS...|                LIMA| 1552867517.84|
|MUNICIPALIDAD DIS...|              ANCASH| 1477164267.22|
|MUNICIPALIDAD DIS...|            AREQUIPA| 1452805772.52|
|MUNICIPALIDAD DIS...|            AREQUIPA| 1452655054.83|
|MUNICIPALIDAD DIS...|                LIMA| 1357722238.51|
+--------------------+--------------------+--------------+
only showing top 10 rows


In [11]:
# Ejecución por departamento
mart_geo = spark.read.parquet(f"{gold_path}/MART_INGRESOS_GEOGRAFICO.parquet")
mart_geo.groupBy("Departamento").agg(
    F.sum("MontoPIA").alias("PIA"),
    F.sum("MontoPIM").alias("PIM"),
    F.sum("MontoRecaudado").alias("Recaudado"),
    F.avg("PctEjecucion").alias("PctPromedio")
).orderBy(F.col("Recaudado").desc()).toPandas()

,Departamento,PIA,PIM,Recaudado,PctPromedio
0,LIMA,21873085316,31304423541,33918612889.13,357.321648
1,CUSCO,11060215147,18897026490,19950059517.92,18.804447
2,ANCASH,6827563427,16560280898,17323594417.16,-6.442744
3,AREQUIPA,5336458292,12303438591,14315506331.02,7.319114
4,PIURA,5171027390,10910035325,10020538671.38,1408.026461
5,LA LIBERTAD,4238639025,8919581641,8776164901.80,21.328193
6,CAJAMARCA,3993297099,7565295682,7218776062.83,495.468664
7,ICA,3320768616,6983573006,6743726645.52,12.579459
8,PUNO,3354354059,5774361433,5817493844.63,11.421899
9,JUNIN,2788655378,5298632274,5558482860.88,11.160198


In [13]:
# Deuda predial por municipio
mart_predial = spark.read.parquet(f"{gold_path}/MART_PREDIAL.parquet")
mart_predial.filter(F.col("PreguntaDescripcion").contains("deuda")).groupBy("Municipalidad").agg(
    F.avg("ValorNumerico").alias("DeudaPromedio")
).orderBy(F.col("DeudaPromedio").desc()).show(10)

+--------------------+---------------+
|       Municipalidad|  DeudaPromedio|
+--------------------+---------------+
|MUNICIPALIDAD DIS...|30707282.878261|
|MUNICIPALIDAD DIS...|12130020.227000|
|MUNICIPALIDAD DIS...| 6833249.703250|
|MUNICIPALIDAD DIS...| 5822655.744000|
|MUNICIPALIDAD DIS...| 3896098.131143|
|MUNICIPALIDAD DIS...| 2656811.150250|
|MUNICIPALIDAD DIS...| 2123712.349750|
|MUNICIPALIDAD DIS...| 2040991.070290|
|MUNICIPALIDAD DIS...| 1912012.659250|
|MUNICIPALIDAD PRO...| 1496868.289750|
+--------------------+---------------+
only showing top 10 rows


In [14]:
# Cobertura de servicios RENAMU
mart_renamu = spark.read.parquet(f"{gold_path}/MART_RENAMU.parquet")
mart_renamu.filter(F.col("EsAfirmativo") == True).groupBy("Departamento", "Descripcion").agg(
    F.count("*").alias("MunicipiosConServicio")
).orderBy("Departamento", F.col("MunicipiosConServicio").desc()).show(20)

+------------+-----------+---------------------+
|Departamento|Descripcion|MunicipiosConServicio|
+------------+-----------+---------------------+
|    AMAZONAS|    VFI_P61|                  420|
|    AMAZONAS|    VFI_P85|                  420|
|    AMAZONAS|    VFI_P14|                  420|
|    AMAZONAS|    VFI_P19|                  420|
|    AMAZONAS|    VFI_P79|                  420|
|    AMAZONAS|    VFI_P62|                  420|
|    AMAZONAS|    VFI_P46|                  420|
|    AMAZONAS|    VFI_P80|                  420|
|    AMAZONAS|    VFI_P67|                  420|
|    AMAZONAS|    VFI_P30|                  420|
|    AMAZONAS|    VFI_P13|                  420|
|    AMAZONAS|     P60A_1|                  420|
|    AMAZONAS|    VFI_P23|                  420|
|    AMAZONAS|    VFI_C97|                  420|
|    AMAZONAS|    VFI_P69|                  420|
|    AMAZONAS|    VFI_P16|                  420|
|    AMAZONAS|    VFI_P38|                  420|
|    AMAZONAS|    VF